In [ ]:
# Import required libraries for data manipulation, preprocessing, and model evaluation.

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
)



In [ ]:
# Upload housing_iteration_5_classification.csv to your own Google Drive
# Paste your file's ID below
file_id = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"
url = f"https://drive.google.com/uc?export=download&id={file_id}"
df = pd.read_csv(url)
df = df.set_index("Id")

In [ ]:
# Drops redundant or low-impact features.

X = df.drop(columns= [
    "MoSold", "GarageYrBlt", "LowQualFinSF", "Electrical",
    "BsmtUnfSF", "BsmtFinSF2", "BsmtFinType2", "BsmtFinSF1",
    "MasVnrArea", "Exterior2nd", "BldgType", "Condition2",
    "LandSlope", "Utilities", "LotShape", "Alley",
])

# Separates the target label('Expensive') from the feature matrix.
y = X.pop("Expensive")

# -----------------------------
# Feature Engineering
# -----------------------------
# Converts MSSubClass to string type for One-Hot Encoding, as these numeric codes are not hierarchical.
X["MSSubClass"] = X["MSSubClass"].astype(str)

In [ ]:
# Splits the dataset into training (80%) and testing (20%) sets.

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=123)


In [ ]:
# -----------------------------
# Preprocessing Pipelines
# -----------------------------
# Identify numerical, nominal and ordinal columns

num_cols = [
    "YrSold", "MiscVal", "PoolArea", "ScreenPorch", "3SsnPorch",
    "EnclosedPorch", "OpenPorchSF", "WoodDeckSF", "GarageArea",
    "GarageCars", "Fireplaces", "TotRmsAbvGrd", "KitchenAbvGr",
    "BedroomAbvGr", "HalfBath", "FullBath", "BsmtHalfBath",
    "BsmtFullBath", "GrLivArea", "1stFlrSF", "2ndFlrSF",
    "TotalBsmtSF", "YearRemodAdd", "YearBuilt", "OverallCond",
    "OverallQual", "LotArea", "LotFrontage",
]


nominal_cols = [
    "SaleCondition", "SaleType", "MiscFeature", "Fence", "GarageType",
    "CentralAir", "Heating", "Foundation", "MasVnrType", "Exterior1st",
    "RoofMatl", "RoofStyle", "HouseStyle", "Neighborhood", "LotConfig",
    "LandContour", "Street", "MSZoning",
    "Condition1",
    "MSSubClass",
]

ordinal_cols = [
    "PoolQC", "PavedDrive", "GarageCond", "GarageQual", "GarageFinish",
    "FireplaceQu", "Functional", "KitchenQual", "HeatingQC",
    "BsmtFinType1", "BsmtExposure", "BsmtCond", "BsmtQual",
    "ExterCond", "ExterQual",
]

# For ordinal encoding, specify its explicit order

ordinal_categories = [
    ["N_A", "Fa", "TA", "Gd", "Ex"],                       # PoolQC
    ["N", "P", "Y"],                                       # PavedDrive
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # GarageCond
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # GarageQual
    ["N_A", "Unf", "RFn", "Fin"],                           # GarageFinish
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # FireplaceQu
    ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],  # Functional
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # KitchenQual
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # HeatingQC
    ["N_A", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],      # BsmtFinType1
    ["N_A", "No", "Mn", "Av", "Gd"],                        # BsmtExposure
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # BsmtCond
    ["N_A", "Po", "Fa", "TA", "Gd", "Ex"],                  # BsmtQual
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # ExterCond
    ["Po", "Fa", "TA", "Gd", "Ex"],                         # ExterQual
]

In [ ]:
# Numerical pipeline: imputes missing values using the mean strategy.
num_pipe = make_pipeline(SimpleImputer(strategy="mean"))

# Nominal pipeline: imputes missing values with "N_A" and applies One-Hot Encoding.
nominal_pipe = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="N_A"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)
# Ordinal pipeline: imputes missing values with "N_A" and encodes ordinal features numerically.
ordinal_pipe = make_pipeline(
    SimpleImputer(strategy="constant", fill_value="N_A"),
    OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1),
)
# Combines numerical, ordinal, and nominal pipelines into a single preprocessor.
preprocessor = make_column_transformer(
    (num_pipe, num_cols),
    (ordinal_pipe, ordinal_cols),
    (nominal_pipe, nominal_cols),
)

In [ ]:
# -----------------------------
# Random Forest + GridSearchCV
# -----------------------------

# Full pipeline: integrates feature preprocessing with a Random Forest classifier.
random_forest_pipe = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=123),
)

# Parameter grid for Random Forest hyperparameter tuning.
rf_param_grid = {
    "randomforestclassifier__n_estimators": [100, 200, 400],
    "randomforestclassifier__max_depth": [5, 10, 20, None],
    "randomforestclassifier__min_samples_leaf": [1, 3, 5],
}

# Grid search setup with 5-fold cross-validation for Random Forest tuning.
# n_jobs=-1: Parallelizes search using all available CPU cores to speed up execution
# verbose=1: Displays basic progress logs showing total fits and completion status

rf_search = GridSearchCV(
    random_forest_pipe,
    rf_param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
)

# Fits the Grid Search model on the training data.
rf_search.fit(X_train, y_train)

print("Best parameters:", rf_search.best_params_)
print("Best CV accuracy:", round(rf_search.best_score_, 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best parameters: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 400}
Best CV accuracy: 0.9461


In [ ]:
# Generates Random Forest predictions on unseen test data using the best model.
y_pred = rf_search.predict(X_test)

print("Accuracy (Random Forest):", round(accuracy_score(y_test, y_pred), 4))
print("Recall (Random Forest):", round(recall_score(y_test, y_pred), 4))
print("Precision (Random Forest):", round(precision_score(y_test, y_pred), 4))
print("F1 Score: (Random Forest)", round(f1_score(y_test, y_pred), 4))
print("Balanced Accuracy: (Random Forest)", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Cohen's Kappa: (Random Forest)", round(cohen_kappa_score(y_test, y_pred), 4))
print("Confusion matrix (Random Forest) :\n", confusion_matrix(y_test, y_pred))

Accuracy (Random Forest): 0.9623
Recall (Random Forest): 0.8095
Precision (Random Forest): 0.9189
F1 Score: (Random Forest) 0.8608
Balanced Accuracy: (Random Forest) 0.8988
Cohen's Kappa: (Random Forest) 0.8391
Confusion matrix (Random Forest) :
 [[247   3]
 [  8  34]]


In [ ]:
# -----------------------------
# Logistic Regression + GridSearchCV
# -----------------------------
# Create a pipeline combining preprocessing, feature scaling, and Logistic Regression
# MinMaxScaler(): Scales all numerical features to the range [0, 1] to prevent larger-magnitude features from dominating the gradient-based model
# max_iter=5000: Increases the maximum solver iterations from default (100) to 5000 to guarantee model convergence without hitting iteration limits

logistic_reg_pipe = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    LogisticRegression(max_iter=5000, random_state=123),
)


# Define hyperparameter grid for GridSearchCV to search for the best regularization strength (C)
# Smaller C (0.01) increases regularization to prevent overfitting; larger C (100) relaxes it

logreg_param_grid = {
    "logisticregression__C": [0.01, 0.1, 1, 10, 100],
}


# Set up 5-fold cross-validation grid search to find the optimal C parameter for the Logistic Regression pipeline
# cv=5: Evaluates each candidate C value using 5-fold stratified cross-validation
# n_jobs=-1: Parallelizes search using all available CPU cores to speed up execution
# verbose=1: Displays basic progress logs showing total fits and completion status

logreg_search = GridSearchCV(
    logistic_reg_pipe,
    logreg_param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
)

# Fits the pipeline (preprocessor, MinMaxScaler, LogisticRegression) on training data to identify the best hyperparameter configuration
logreg_search.fit(X_train, y_train)


# Print the optimal hyperparameter combination found during grid search
print("Best parameters:", logreg_search.best_params_)

# Print the highest cross-validation score achieved by the best model config, rounded to 4 decimals
print("Best CV accuracy:", round(logreg_search.best_score_, 4))

# Generates Logistic Regression predictions on unseen test data using the best model.
y_pred = logreg_search.predict(X_test)

print("Accuracy (Logistic Regression):", round(accuracy_score(y_test, y_pred), 4))
print("Recall (Logistic Regression):", round(recall_score(y_test, y_pred), 4))
print("Precision (Logistic Regression):", round(precision_score(y_test, y_pred), 4))
print("F1 Score (Logistic Regression) :", round(f1_score(y_test, y_pred), 4))
print("Balanced Accuracy (Logistic Regression) :", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Cohen's Kappa (Logistic Regression) :", round(cohen_kappa_score(y_test, y_pred), 4))
print("Confusion matrix (Logistic Regression):\n", confusion_matrix(y_test, y_pred))

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best parameters: {'logisticregression__C': 1}
Best CV accuracy: 0.9469
Accuracy (Logistic Regression): 0.9726
Recall (Logistic Regression): 0.8571
Precision (Logistic Regression): 0.9474
F1 Score (Logistic Regression) : 0.9
Balanced Accuracy (Logistic Regression) : 0.9246
Cohen's Kappa (Logistic Regression) : 0.8842
Confusion matrix (Logistic Regression):
 [[248   2]
 [  6  36]]


In [ ]:
# -----------------------------
# Support Vector Machine (SVM) + GridSearchCV
# -----------------------------

# Construct Support Vector Machine pipeline with preprocessing and feature scaling
# MinMaxScaler(): Scales features to [0, 1] as distance-based SVM algorithms are highly sensitive to feature scales
svm_pipe = make_pipeline(
    preprocessor,
    MinMaxScaler(),
    SVC(random_state=123),
)

# Define hyperparameter grid for SVM tuning
# svc__C: Regularization strength; smaller values increase margin width, larger values penalize misclassifications
# svc__kernel: Decision boundary type ('linear' for straight boundary, 'rbf' for non-linear decision boundary)
# svc__gamma: Kernel coefficient for 'rbf'; controls the radius of influence for single training points
svm_param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"],
}

# Configure 5-fold cross-validation grid search using all CPU cores (n_jobs=-1) with basic progress logging (verbose=1)
svm_search = GridSearchCV(
    svm_pipe,
    svm_param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
)

# Fit grid search on training data across all parameter combinations and cross-validation folds
svm_search.fit(X_train, y_train)



# Display best hyperparameter configuration and cross-validation accuracy score
print()
print("Best parameters:", svm_search.best_params_)
print("Best CV accuracy:", round(svm_search.best_score_, 4))

# Generate predictions on the unseen test set using the best tuned SVM model
y_pred = svm_search.predict(X_test)

# Evaluate classification performance metrics on the test dataset

print("Accuracy (SVM):", round(accuracy_score(y_test, y_pred), 4))
print("Recall (SVM):", round(recall_score(y_test, y_pred), 4))
print("Precision (SVM):", round(precision_score(y_test, y_pred), 4))
print("F1 Score (SVM):", round(f1_score(y_test, y_pred), 4))
print("Balanced Accuracy (SVM):", round(balanced_accuracy_score(y_test, y_pred), 4))
print("Cohen's Kappa (SVM):", round(cohen_kappa_score(y_test, y_pred), 4))
print("Confusion matrix (SVM):\n", confusion_matrix(y_test, y_pred))


Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best parameters: {'svc__C': 0.1, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
Best CV accuracy: 0.9426
Accuracy (SVM): 0.9486
Recall (SVM): 0.7619
Precision (SVM): 0.8649
F1 Score (SVM): 0.8101
Balanced Accuracy (SVM): 0.871
Cohen's Kappa (SVM): 0.7806
Confusion matrix (SVM):
 [[245   5]
 [ 10  32]]


In [ ]:
# Downloads and prepares the external test set for the classroom competition.
# Upload test_set.csv to your own Google Drive
# Paste your file's ID below
file_id1 = "YOUR_GOOGLE_DRIVE_FILE_ID_HERE"
url1 = f"https://drive.google.com/uc?export=download&id={file_id1}"
testing_data = pd.read_csv(url1)
testing_data = testing_data.set_index('Id')

In [ ]:
#------- Random Forest prediction for classroom competetion---------
# Feature Engineering again for "MSSubClass" column
testing_data["MSSubClass"] = testing_data["MSSubClass"].astype(str)

# Generates Random Forest predictions on the unlabelled competition dataset and stores them as a new column.
testing_data["Expensive"] = rf_search.predict(testing_data)

# Exports the "Expensive" prediction column to a CSV file for competition submission.
testing_data['Expensive'].to_csv('./submission_randomforest.csv')

# Save and download the Random Forest predictions for submission
from google.colab import files
files.download('./submission_randomforest.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#------- Logistic Regression prediction for classroom competition---------
# Feature Engineering again for "MSSubClass" column
testing_data["MSSubClass"] = testing_data["MSSubClass"].astype(str)

# Generates Logistic Regression predictions on the unlabelled competition dataset and stores them as a new column.
testing_data["Expensive"] = logreg_search.predict(testing_data)

# Exports the "Expensive" prediction column to a CSV file for competition submission.
testing_data['Expensive'].to_csv('./submission_logreg.csv')

# Save and download the Logistic Regression predictions for submission
from google.colab import files
files.download('./submission_logreg.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#------- Support Vector Machine prediction for classroom competition---------
# Feature Engineering again for "MSSubClass" column
testing_data["MSSubClass"] = testing_data["MSSubClass"].astype(str)

# Generates Support Vector Machine predictions on the unlabelled competition dataset and stores them as a new column.
testing_data["Expensive"] = svm_search.predict(testing_data)

# Exports the "Expensive" prediction column to a CSV file for competition submission.
testing_data['Expensive'].to_csv('./submission_svm.csv')

# Save and download the Support Vector Machine predictions for submission
from google.colab import files
files.download('./submission_svm.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>